# Bates--Hawkes stationary proxy calibration

This notebook calibrates the documented stationary-intensity proxy on the
current local-only LSE sample. It is a
benchmark distinct from the exact event-dependent affine model.


In [ ]:
import json
from pathlib import Path

from BatesHawkes import BatesHawkes
from calibration_workflow import (
    load_calibration_surface, option_diagnostics, plot_residuals,
    plot_smiles, price_surface,
)

DATA = Path("Data/lse_local")
SEED = 20260811
DIVIDEND_YIELD = 0.0
surface, spot = load_calibration_surface(DATA, DIVIDEND_YIELD)
surface.head()


In [ ]:
report = BatesHawkes.calibrate_bates_hawkes_proxy(
    surface,
    spot,
    q=DIVIDEND_YIELD,
    seed=SEED,
    pricing="cos",
    return_report=True,
)
report.as_dict()


In [ ]:
parameters = report.x
model_prices = price_surface(
    surface,
    lambda strikes, maturity, rate: BatesHawkes.prices_proxy_cos(
        spot, strikes, maturity, *parameters, rate, DIVIDEND_YIELD
    ),
)
diagnostics = option_diagnostics(
    surface, spot, model_prices, DIVIDEND_YIELD
)
diagnostics.to_csv(DATA / "bates_hawkes_proxy_diagnostics.csv", index=False)
(DATA / "bates_hawkes_proxy_report.json").write_text(
    json.dumps(report.as_dict(), indent=2), encoding="utf-8"
)
diagnostics.head()


In [ ]:
plot_smiles(
    diagnostics,
    "Bates--Hawkes proxy",
    DATA / "bates_hawkes_proxy_volatility_smile.png",
)
plot_residuals(
    diagnostics,
    "Bates--Hawkes proxy",
    DATA / "bates_hawkes_proxy_residual_heatmap.png",
)
